# Player KPI Correlation Analysis
**Goal:** Use a correlation heatmap to surface the strongest relationships between player behavior and spending — then use those findings to write targeted SQL queries.

**Data source:** `heatmap_data.csv` — one row per player, aggregated from MySQL using `heatmap_join.sql`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('heatmap_data.csv')

print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')
df.head()

## 1. Data Overview

In [ ]:
print('=== COLUMN TYPES ===')
print(df.dtypes)
print('\n=== NULL COUNTS ===')
print(df.isnull().sum())
print('\n=== KEY STATS ===')
df.describe().round(2)

## 2. Prepare for Correlation

In [ ]:
numeric_df = df.drop(columns=['user_id'])
print('Columns going into heatmap:')
print(numeric_df.columns.tolist())

## 3. Full Correlation Heatmap — All Columns

In [ ]:
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(18, 15))

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 8},
    ax=ax
)

ax.set_title('Player Behavior & Spending — Correlation Matrix', fontsize=15, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig('heatmap_full.png', dpi=300, bbox_inches='tight')
plt.savefig('heatmap_full.svg', bbox_inches='tight')
plt.show()
print('Saved → heatmap_full.png + heatmap_full.svg')

## 4. Top Correlations — Ranked

In [ ]:
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']
corr_pairs['abs_corr'] = corr_pairs['correlation'].abs()
corr_pairs = corr_pairs.sort_values('abs_corr', ascending=False)

print('=== TOP 20 STRONGEST CORRELATIONS ===')
print(corr_pairs.head(20).to_string(index=False))

## 5. Focused Heatmap — Spending Signals Only

In [ ]:
spend_cols = ['total_spend', 'transaction_count', 'avg_spend_per_txn']
behavior_cols = [c for c in numeric_df.columns if c not in spend_cols]

spend_corr = numeric_df[behavior_cols + spend_cols].corr().loc[behavior_cols, spend_cols]

fig, ax = plt.subplots(figsize=(8, 10))

sns.heatmap(
    spend_corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 10},
    ax=ax
)

ax.set_title('What Predicts Spending?\nBehavior vs. Spend Metrics', fontsize=13, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=30, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)

plt.tight_layout()
plt.savefig('heatmap_spend_focus.png', dpi=300, bbox_inches='tight')
plt.savefig('heatmap_spend_focus.svg', bbox_inches='tight')
plt.show()
print('Saved → heatmap_spend_focus.png + heatmap_spend_focus.svg')

## 6. Focused Heatmap — Retention Signals Only

In [ ]:
retention_corr = numeric_df.corr()[['avg_retention']].drop('avg_retention')
retention_corr = retention_corr.sort_values('avg_retention', ascending=False)

fig, ax = plt.subplots(figsize=(5, 10))

sns.heatmap(
    retention_corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 11},
    ax=ax
)

ax.set_title('What Predicts Retention?', fontsize=13, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=0, labelsize=11)
ax.tick_params(axis='y', rotation=0, labelsize=10)

plt.tight_layout()
plt.savefig('heatmap_retention_focus.png', dpi=300, bbox_inches='tight')
plt.savefig('heatmap_retention_focus.svg', bbox_inches='tight')
plt.show()
print('Saved → heatmap_retention_focus.png + heatmap_retention_focus.svg')

## 7. Findings Summary

| Relationship | Correlation | Business Question for SQL |
|---|---|---|
| e.g. sessions_per_week ↔ total_spend | 0.XX | Do high-frequency players spend more? |
| e.g. avg_rank_score ↔ avg_retention | 0.XX | Do higher-ranked players retain better? |
| ... | ... | ... |

> **Next step:** Take the 3–5 strongest non-obvious correlations and write a SQL query for each one.